[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eo-agh/data-analysis-earth-sciences/blob/main/docs/embeddings.ipynb)

## Earth Embeddings

**Embedding** (reprezentacja wektorowa) to odwzorowanie danych wejściowych - obrazu, tekstu, lokalizacji - na zwartą, wielowymiarową przestrzeń liczbową (np. wektor 512 lub 1024 floatów). W tej przestrzeni semantycznie podobne obiekty leżą blisko siebie, a różne - daleko.

W przetwarzaniu języka naturalnego (NLP) embeddingi zrewolucjonizowały wyszukiwanie semantyczne i klasyfikację tekstu (Word2Vec, BERT, GPT). Analogicznie, w obserwacji Ziemi **embeddingi obrazów satelitarnych** kodują cechy spektralne, przestrzenne i czasowe danej sceny w jeden gęsty wektor - bez ręcznego projektowania cech ani obliczania wskaźników takich jak NDVI.

Dzięki embeddingom EO możliwe jest:

- **Wyszukiwanie treściowe** (*content-based image retrieval*) - "znajdź wszystkie sceny podobne do tej", bez potrzeby etykietowania danych.
- **Transfer learning** - fine-tuning małej sieci na zadanie downstream (klasyfikacja, detekcja zmian, segmentacja semantyczna) przy minimalnej liczbie etykiet.
- **Analiza wieloczasowa** - porównywanie embeddingów tej samej lokalizacji z różnych dat w celu wykrycia zmian.
- **Klasteryzacja** obszarów o podobnej charakterystyce środowiskowej lub użytkowaniu terenu.

## Geospatial Foundation Models

Analogicznie do dużych modeli językowych (LLM), w obserwacji Ziemi pojawiły się **geospatial foundation models** - modele wstępnie wytrenowane na ogromnych zbiorach danych satelitarnych, produkujące wysokiej jakości reprezentacje nadające się do różnych zadań downstream.

| Model | Organizacja | Architektura | Dane treningowe | Wymiar embeddingu |
|---|---|---|---|---|
| **Clay v1.5** | Made with Clay | ViT + MAE | Sentinel-2/1, Landsat, NAIP, MODIS | 1024 |
| **Prithvi-EO-2.0** | NASA + IBM | ViT (TerraTorch) | HLS (Landsat + S-2, 30 m) | 768 |
| **SatCLIP** | Microsoft Research | CLIP (ViT + geoencoder) | Sentinel-2 + współrzędne GPS | 512 |
| **RemoteCLIP** | BAAI | CLIP | Obrazy EO + opisy tekstowe | 512 |

Wszystkie modele i ich wagi są publicznie dostępne na [Hugging Face](https://huggingface.co/made-with-clay).

### Architektura Clay - ViT + MAE

Clay używa architektury **Vision Transformer (ViT)** trenowanej metodą **Masked Autoencoder (MAE)**: podczas treningu losowo maskuje się fragmenty obrazu satelitarnego, a model musi je zrekonstruować. Dzięki temu uczy się bogatych reprezentacji *bez potrzeby etykietowania danych* (self-supervised learning).

Model przyjmuje na wejściu **chip** obrazu (256×256 px), informację o długościach fal pasm spektralnych (`wavelengths`), timestamp oraz lokalizację - a na wyjściu zwraca wektor 1024-wymiarowy.

In [ ]:
# pip install duckdb pandas numpy matplotlib scikit-learn
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## Gotowe embeddingi Clay v1.5 - Sentinel-2

Uruchomienie modelu Clay wymaga GPU i pobrania wag (~1 GB). Na potrzeby eksploracji Clay opublikował **pre-computed embeddingi** Sentinel-2 dla wybranych regionów świata na platformie [Source Cooperative](https://source.coop/clay/clay-v1-5-sentinel2) - w formacie Parquet.

Każdy wiersz to jeden chip 256×256 px z następującymi kolumnami:
- `bbox` - bounding box w WGS84,
- `date` - data akwizycji,
- `embedding` - lista 1024 wartości float32.

In [ ]:
parquet_url = "https://data.source.coop/clay/clay-v1-5-sentinel2/suriname/embeddings.parquet"

con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{parquet_url}') LIMIT 2000").df()

print(f"Liczba chipów  : {len(df)}")
print(f"Kolumny        : {df.columns.tolist()}")
df.head(3)

In [ ]:
# Kolumna 'embedding' zawiera listę 1024 floatów - zamieniamy na macierz numpy
embeddings = np.stack(df["embedding"].values)
print(f"Kształt macierzy embeddingów: {embeddings.shape}")
print(f"dtype: {embeddings.dtype}")

## Redukcja wymiarowości - PCA

Przestrzeń 1024-wymiarową rzutujemy na 2D przy użyciu **PCA** (Principal Component Analysis). Pozwala to zobrazować strukturę embeddingów - potencjalnie odkrywając klastry odpowiadające różnym typom pokrycia terenu (lasy, wody, obszary zabudowane).

Do bardziej nieliniowych struktur można zastosować **UMAP** (`pip install umap-learn`), który często ujawnia lepiej rozdzielone klastry.

In [ ]:
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(embeddings)

print(f"Wyjaśniona wariancja PC1 + PC2: {pca.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=6, alpha=0.5,
                c=emb_2d[:, 0], cmap="viridis")
plt.colorbar(sc, ax=ax, label="PC1")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Embeddingi Clay v1.5 (Sentinel-2, Surinam) - projekcja PCA")
plt.tight_layout()
plt.show()

## Wyszukiwanie podobnych lokalizacji

Embeddingi umożliwiają **semantyczne wyszukiwanie obrazów** - dla danego chipu (query) znajdujemy inne miejsca o podobnej charakterystyce, mierząc **odległość cosinusową** w przestrzeni embeddingów.

Odległość cosinusowa mierzy kąt między wektorami (wartości od 0 = identyczne do 1 = ortogonalne), co sprawia, że jest odporna na różnice w skali embeddingów.

In [ ]:
rng = np.random.default_rng(42)
query_idx = int(rng.integers(0, len(embeddings)))
query = embeddings[query_idx].reshape(1, -1)

sims = cosine_similarity(query, embeddings)[0]
# Pomijamy samego siebie (indeks 0 po sortowaniu malejącym)
top5_idx = np.argsort(sims)[::-1][1:6]

print(f"Query chip index : {query_idx}")
if "date" in df.columns:
    print(f"Query chip date  : {df['date'].iloc[query_idx]}")

print("\n5 najbardziej podobnych chipów:")
for rank, idx in enumerate(top5_idx, 1):
    date_str = str(df['date'].iloc[idx]) if 'date' in df.columns else ''
    print(f"  #{rank}  index={idx:4d}  similarity={sims[idx]:.4f}  {date_str}")

## Uruchomienie modelu Clay (opcjonalnie - wymaga GPU)

Jeśli chcemy wygenerować własne embeddingi dla dowolnych danych Sentinel-2, możemy załadować wagi modelu z Hugging Face i uruchomić enkoder bezpośrednio:

```python
# pip install clay  (instaluje claymodel i zależności)
import torch
import yaml
from claymodel.module import ClayMAEModule

# Pobranie checkpointu z Hugging Face Hub
from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(repo_id="made-with-clay/Clay", filename="clay-v1.5.ckpt")

model = ClayMAEModule.load_from_checkpoint(ckpt_path)
model.eval()

# chips:      (batch, bands, height, width)  np. (4, 10, 256, 256) dla Sentinel-2
# timestamps: (batch, 4) - rok, miesiąc, dzień, godzina
# wavelengths:(batch, bands) - długości fal w nm
chips      = torch.randn(4, 10, 256, 256)
timestamps = torch.tensor([[2023, 7, 15, 10]] * 4, dtype=torch.float32)
wavelengths = torch.tensor([[443, 492, 560, 665, 704, 740, 783, 833, 865, 945]] * 4,
                            dtype=torch.float32)

with torch.no_grad():
    embeddings = model.encoder(chips, timestamps, wavelengths)

print(embeddings.shape)  # torch.Size([4, 1024])
```